In [ ]:
import pandas as pd
import numpy as np

In [ ]:

# Instellingen om te tweaken
# Aantal aanbevelingen; bepaalt ook de K in Precision@K en NDCG@K.
TOP_K = 3
# Maximumaantal meest vergelijkbare studenten dat voor voorspellingen wordt gebruikt.
N_NEIGHBORS = 10
# Minimumaantal gedeelde materialen waarop een studentcorrelatie wordt berekend.
MIN_COMMON_ITEMS = 2
# Minimumaantal gekozen buren dat een materiaal moet hebben beoordeeld.
MIN_ITEM_SUPPORT = 2
# Vanaf deze intscore telt een achtergehouden materiaal als relevant.
RELEVANCE_THRESHOLD = 4
# Deel van de interactieparen per student dat als testdata wordt achtergehouden.
TEST_FRACTION = 0.2
# Seed voor een reproduceerbare train/test-split.
RANDOM_STATE = 42
# True gebruikt alleen positief gecorreleerde buren; False laat ook andere correlaties toe.
ONLY_POSITIVE_NEIGHBORS = True
# Kies componenten voor de interactiescore door regels toe te voegen of weg te laten.
# De gewichten geven het relatieve belang; ze worden automatisch genormaliseerd.
SCORE_WEIGHTS = {
    # 'leesprc': 0.15,      # Afgeleid van leespercentage (omgezet naar een 0-5-schaal).
    'inttype': 0.25,      # Afgeleid van interactietype via eval_type_scores hieronder.
    'beoordeling': 0.75,  # De beoordeling uit de interactiedata.
}

# Lees de opgeschoonde data en bereken de interactiescore, onafhankelijk van de baselinecellen.
eval_raw = pd.read_csv('./dataset/cleaned_Interacties.csv')

eval_type_scores = {
    'geliket': 5.0,
    'opgeslagen': 4.0,
    'gelezen': 3.0,
    'bekeken': 2.0,
    'niet_afgemaakt': 1.0,
    'afgewezen': 1.0,
}
eval_unknown_types = (
    set(eval_raw['interactietype'].dropna().unique()) - set(eval_type_scores)
)
if eval_unknown_types:
    raise ValueError('Onbekende interactietypes: {}'.format(sorted(eval_unknown_types)))

eval_all_inputs = pd.DataFrame({
    'leesprc': (eval_raw['leespercentage'] / 20).round(),
    'inttype': eval_raw['interactietype'].map(eval_type_scores),
    'beoordeling': pd.to_numeric(eval_raw['beoordeling'], errors='coerce'),
})
if not SCORE_WEIGHTS:
    raise ValueError('SCORE_WEIGHTS moet minstens één scorecomponent bevatten.')
eval_unknown_components = set(SCORE_WEIGHTS) - set(eval_all_inputs.columns)
if eval_unknown_components:
    raise ValueError('Onbekende scorecomponenten: {}'.format(sorted(eval_unknown_components)))
if any(weight <= 0 for weight in SCORE_WEIGHTS.values()):
    raise ValueError('Alle gekozen scoregewichten moeten groter zijn dan 0.')
eval_inputs = eval_all_inputs[list(SCORE_WEIGHTS)]
eval_weights = pd.Series(SCORE_WEIGHTS)
eval_weighted_sum = eval_inputs.mul(eval_weights, axis='columns').sum(axis=1, min_count=1)
eval_available_weight = eval_inputs.notna().mul(eval_weights, axis='columns').sum(axis=1)
eval_raw['intscore'] = eval_weighted_sum / eval_available_weight

# Gemiddelde score per student-materiaalpaar; voorkom overlap tussen train en test.
eval_pairs = (
    eval_raw.groupby(['student_id', 'cataloguscode'], as_index=False)['intscore']
    .mean()
    .dropna(subset=['intscore'])
    .reset_index(drop=True)
)

# Willekeurige holdout per student; houd minstens één paar in de trainingsset.
eval_rng = np.random.default_rng(RANDOM_STATE)
eval_test_indices = []
for _, eval_user_pairs in eval_pairs.groupby('student_id', sort=False):
    if len(eval_user_pairs) < 2:
        continue
    eval_n_test = max(1, round(len(eval_user_pairs) * TEST_FRACTION))
    eval_n_test = min(eval_n_test, len(eval_user_pairs) - 1)
    eval_seed = int(eval_rng.integers(0, 2**32 - 1))
    eval_test_indices.extend(
        eval_user_pairs.sample(n=eval_n_test, random_state=eval_seed).index
    )
if not eval_test_indices:
    raise ValueError('Niet genoeg student-materiaalparen voor de train/test-split.')

eval_test = eval_pairs.loc[eval_test_indices].copy()
eval_train = eval_pairs.drop(index=eval_test_indices).copy()
eval_students = eval_pairs['student_id'].unique()
eval_items = eval_pairs['cataloguscode'].unique()
eval_train_matrix = (
    eval_train.pivot(index='student_id', columns='cataloguscode', values='intscore')
    .reindex(index=eval_students, columns=eval_items)
)
eval_similarity = eval_train_matrix.T.corr(min_periods=MIN_COMMON_ITEMS)

def eval_recommend(user_id):
    eval_neighbors = eval_similarity.loc[user_id].drop(index=user_id).dropna()
    if ONLY_POSITIVE_NEIGHBORS:
        eval_neighbors = eval_neighbors[eval_neighbors > 0]
    eval_neighbors = eval_neighbors.nlargest(N_NEIGHBORS)
    if eval_neighbors.empty:
        return []

    eval_neighbor_scores = eval_train_matrix.loc[eval_neighbors.index]
    eval_weighted_sum = eval_neighbor_scores.mul(eval_neighbors, axis=0).sum(axis=0, min_count=1)
    eval_weight_sum = eval_neighbor_scores.notna().mul(eval_neighbors, axis=0).sum(axis=0)
    eval_predicted = eval_weighted_sum / eval_weight_sum
    eval_support = eval_neighbor_scores.notna().sum(axis=0)
    eval_unseen = eval_train_matrix.loc[user_id].isna()
    eval_candidates = eval_unseen & (eval_support >= MIN_ITEM_SUPPORT)
    return eval_predicted[eval_candidates].nlargest(TOP_K).index.tolist()

# Een testinteractie is relevant als de zelf berekende score de grens haalt.
eval_relevant_items = (
    eval_test.loc[eval_test['intscore'] >= RELEVANCE_THRESHOLD]
    .groupby('student_id')['cataloguscode']
    .apply(set)
)
if eval_relevant_items.empty:
    raise ValueError('Geen relevante testitems; controleer RELEVANCE_THRESHOLD.')

eval_precision_name = 'Precision@{}'.format(TOP_K)
eval_ndcg_name = 'NDCG@{}'.format(TOP_K)
eval_results = []
for eval_user_id, eval_relevant in eval_relevant_items.items():
    eval_recs = eval_recommend(eval_user_id)
    eval_hits = [item in eval_relevant for item in eval_recs]
    eval_precision = sum(eval_hits) / TOP_K
    eval_dcg = sum(
        1 / np.log2(rank + 1)
        for rank, is_relevant in enumerate(eval_hits, start=1)
        if is_relevant
    )
    eval_ideal_hits = min(TOP_K, len(eval_relevant))
    eval_idcg = sum(1 / np.log2(rank + 1) for rank in range(1, eval_ideal_hits + 1))
    eval_ndcg = eval_dcg / eval_idcg if eval_idcg else 0.0
    eval_results.append({
        'student_id': eval_user_id,
        eval_precision_name: eval_precision,
        eval_ndcg_name: eval_ndcg,
        'relevant_in_test': len(eval_relevant),
        'recommendations_returned': len(eval_recs),
    })

eval_metrics = pd.DataFrame(eval_results)
eval_component_sources = {
    'leesprc': 'leespercentage / 20, afgerond',
    'inttype': 'interactietype (gescoord via eval_type_scores)',
    'beoordeling': 'beoordeling',
}
print('Gebruikte scorecomponenten en ingestelde relatieve gewichten:')
for eval_component, eval_weight in SCORE_WEIGHTS.items():
    print('  {} <- {}: {}'.format(
        eval_component, eval_component_sources[eval_component], eval_weight
    ))
print('Instellingen: buren={}, min. overlap={}, min. item support={}, relevant vanaf={}'.format(
    N_NEIGHBORS, MIN_COMMON_ITEMS, MIN_ITEM_SUPPORT, RELEVANCE_THRESHOLD
))
print('Aantal geëvalueerde studenten:', len(eval_metrics))
print('Gemiddeld aantal aanbevelingen: {:.2f}'.format(
    eval_metrics['recommendations_returned'].mean()
))
print('Gemiddelde evaluatiescores:')
print(eval_metrics[[eval_precision_name, eval_ndcg_name]].mean().round(4))